THis worksheet simulates 

In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;

Init();

In [ ]:
string proj = "SlipConvergence_Droplet";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();


In [3]:
// int[] kelemSeq = new[] {16};//new int[]{4, 8, 16, 32, 64, 128, 256};
int[] kelemSeq = new int[]{4, 8, 16, 32, 64, 128};
var GridSeq = new IGridInfo[kelemSeq.Length];

In [4]:
for(int iGrid = 0; iGrid < GridSeq.Length; iGrid++) {
    
    int kelem = kelemSeq[iGrid];
    
    GridCommons grd;   
    double[] xNodes = GenericBlas.Linspace(-3.0/2.0, 3.0/2.0, 2 * kelem + 1);
    double[] yNodes = GenericBlas.Linspace(0.0, 3.0/2.0, kelem + 1);    
    grd = Grid2D.Cartesian2DGrid(xNodes, yNodes);

    // HIER ANPASSUNG DER RANDBEDINGUNGEN!
    grd.EdgeTagNames.Add(1, "NavierSlip_linear_ConstantTemperature_upper");
    grd.EdgeTagNames.Add(2, "NavierSlip_linear_ConstantTemperature_lower");
    grd.EdgeTagNames.Add(3, "NavierSlip_linear_ConstantTemperature_left");
    grd.EdgeTagNames.Add(4, "NavierSlip_linear_ConstantTemperature_right");
 
    grd.DefineEdgeTags(delegate (double[] X) {
        byte et = 0;
        if (Math.Abs(X[1] - yNodes.Last()) <= 1.0e-8) // UPPER BOUNDARY
            et = 1;
        if (Math.Abs(X[1] - yNodes.First()) <= 1.0e-8) // LOWER BOUNDARY
            et = 2;
        if (Math.Abs(X[0] - xNodes.First()) <= 1.0e-8) // LEFT BOUNDARY
            et = 3;
        if (Math.Abs(X[0] - xNodes.Last()) <= 1.0e-8) // RIGHT BOUNDARY
            et = 4;
        return et;
    });

    // grd.Name = "StaticDropletOnWall_meshStudy";
    grd.Name = "SlipDropletOnWall_meshStudy_" + kelem;        
    
    BoSSSshell.WorkflowMgm.DefaultDatabase.SaveGrid(ref grd);
    
    GridSeq[iGrid] = grd;
}

In [ ]:
public static List<XNSFE_Control> SlipConvergence_Droplet(double a, double b, int[] pDegS, IGridInfo[] grdS, double[] thetaS, double[] lsS, double[] sigmaS, double[] hvapS){
    
    List<XNSFE_Control> controls = new List<XNSFE_Control>();

    foreach(double theta in thetaS){
        foreach(double ls in lsS){
            foreach(double sigma in sigmaS){
                foreach(double hvap in hvapS){
                    foreach(int pDeg in pDegS){
                        foreach(IGridInfo grd in grdS){
                            controls.Add(SlipConvergence_Droplet(a, b, pDeg, grd, theta, ls, sigma, hvap));
                        }
                    }
                }
            }
        }        
    }
    
    
    return controls;
    
}

public static XNSFE_Control SlipConvergence_Droplet(double a, double b, int pDeg, IGridInfo grd, double theta, double ls, double sigma, double hvap) {
    
    XNSFE_Control C = new XNSFE_Control();
    
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("theta", theta.ToString("N4")));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("hvap", hvap.ToString("N2")));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("interfacesliplength", ls.ToString("N2")));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("sigma", sigma));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("degree", pDeg));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("gridlevel", ((GridCommons)grd).Name.Split('_').Last()));
    
    //C.DbPath = set by workflowMgm during job creation
    C.savetodb = true;
    C.ContinueOnIoError = false;

    C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.SlipDropletLogging());
    C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.Dropletlike());
    C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.MovingContactLineLogging());

    // misc. solver options
    // ====================
    C.NonLinearSolver.MaxSolverIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-6;

    // Level-Set options (AMR)
    // =======================
    C.LSContiProjectionMethod = BoSSS.Solution.LevelSetTools.ContinuityProjectionOption.None;
    C.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.None;
    C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

    // offset of ellipse center to achieve the correct angle
    double y_off = b / (Math.Sqrt(1 + (Math.Pow(a, 2) / Math.Pow(b, 2)) * Math.Pow((Math.Tan(theta/90.0*Math.PI/2.0)), 2)));
    // intersect with the x-axis
    double x0 = a * Math.Cos(Math.Asin(y_off / b));

    Dictionary<byte, string> Boundaries = new Dictionary<byte, string>();
    Boundaries = new Dictionary<byte, string>() {
        { 1, "NavierSlip_linear_ConstantTemperature_upper"},
        { 2, "NavierSlip_linear_ConstantTemperature_lower"},
        { 3, "NavierSlip_linear_ConstantTemperature_left"},
        { 4, "NavierSlip_linear_ConstantTemperature_right"},
    };

    string[] lines;
    if(theta == 90.0) {
        lines = File.ReadAllLines("Temperature_boundarycond90.txt");
    } else if(theta == 80.0) {
        lines = File.ReadAllLines("Temperature_boundarycond80.txt");
    } else {
        throw new NotImplementedException();
    }

    double[] nodes = new double[lines.Length];
    double[] values = new double[lines.Length];
    for (int i = 0; i < lines.Length; i++) {
        string[] xy = lines[i].Split('\t');
        nodes[i] = Convert.ToDouble(xy[0]);
        values[i] = Convert.ToDouble(xy[1]);
    }
    double scale = x0 / nodes.Max();
    nodes = nodes.Select(n => scale * n).ToArray();
    Spline1D TFunc = new Spline1D(nodes, values, 0, Spline1D.OutOfBoundsBehave.Clamp);
    
    C.AddBoundaryValue(Boundaries[1]);
    //C.AddBoundaryValue(Boundaries[2], "Temperature#A", $"X => 1.0/8.0 * Math.Sin(2*Math.PI*(X[0]+{x0})/(2*{x0}))", false);
    C.AddBoundaryValue(Boundaries[2], "Temperature#A", TFunc);
    C.AddBoundaryValue(Boundaries[3]);
    C.AddBoundaryValue(Boundaries[4]);                        

    C.SetDGdegree(pDeg);
    C.SetGrid(grd);

    // Physical Parameters
    // ===================
    C.PhysicalParameters.IncludeConvection = false;
    C.PhysicalParameters.Material = false;
    C.PhysicalParameters.rho_A = 1.0;
    C.PhysicalParameters.rho_B = 0.1; // unequal density
    C.PhysicalParameters.mu_A = 0.5;
    C.PhysicalParameters.mu_B = 0.05;
    C.PhysicalParameters.Sigma = sigma;

    C.PhysicalParameters.betaS_A = 0.0;
    C.PhysicalParameters.betaS_B = 0.0;
    C.PhysicalParameters.betaL = 0.0;
    C.PhysicalParameters.theta_e = theta / 90.0 * Math.PI/2.0;

    C.ThermalParameters.IncludeConvection = false;
    C.IncludeRecoilPressure= false;
    C.ThermalParameters.rho_A = C.PhysicalParameters.rho_A;
    C.ThermalParameters.rho_B = C.PhysicalParameters.rho_B;
    C.ThermalParameters.k_A = 1.0;
    C.ThermalParameters.k_B = 0.1;
    C.ThermalParameters.hVap = hvap; // double.NegativeInfinity implicitly sets a fixed temperature interface without evaporation
    C.ThermalParameters.T_sat = 0.0;

    C.PhysicalParameters.slipI = ls;

    C.LinearSolver = LinearSolverCode.direct_pardiso.GetConfig();
    if(C.LinearSolver is OrthoMGSchwarzConfig) {
        ((OrthoMGSchwarzConfig)C.LinearSolver).CoarseKickIn = 90000;
        ((OrthoMGSchwarzConfig)C.LinearSolver).TargetBlockSize = 10000;
        ((OrthoMGSchwarzConfig)C.LinearSolver).CoarseUsepTG = false;
        ((OrthoMGSchwarzConfig)C.LinearSolver).UsepTG = false;
        ((OrthoMGSchwarzConfig)C.LinearSolver).ConvergenceCriterion = 1e-12;
    }

    C.AddInitialValue("Phi", $"X => ((X[0]).Pow2() /({a}).Pow2() + (X[1]+({y_off})).Pow2() / ({b}).Pow2()) - 1.0", false);
    C.FieldOptions[BoSSS.Solution.NSECommon.VariableNames.LevelSetCGidx(1)].Degree = 2;
    C.AgglomerationThreshold = 0.1;

    // MAKE TIMESTEPPING SETTINGS MORE CLEAR!
    // Timestepping
    // ============
    C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
    C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
    C.Timestepper_BDFinit = BoSSS.Solution.Timestepping.TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = BoSSS.Solution.XdgTimestepping.LevelSetHandling.None;
    
    C.SessionName = ((GridCommons)grd).Name + //grid
        "_P" + pDeg +// degree
        "_CA" + theta.ToString("N2") + // contact angle
        "_SL" + ls.ToString("N2") + // interface slip length
        "_ST" + sigma.ToString("N2") + // surface tension
        "_HV" + hvap.ToString("N2"); // evaporation 
    
    //C.TracingNamespaces = "*";
    
    return C;
}

In [6]:
int[] degS = new int[] { 2, 3, 4 };

In [7]:
int iDeg0 = 0;
int iGrd0 = 0;

In [8]:
List<XNSFE_Control> controls = new List<XNSFE_Control>();

In [ ]:
// Testcase setup
// ==============
bool elliptic  = true;

double radius = 0.8;
double a      = elliptic ? 0.816 : 0.8;
double b      = elliptic ? 0.784 : 0.8;

double[] thetaS;
double[] lsS;
double[] sigmaS;
double[] hvapS;

//stage 1
thetaS            = new double[] {80, 90};
lsS               = new double[] {0.0, 0.1, double.PositiveInfinity};
sigmaS                = new double[] {0.1};
hvapS                = new double[] {double.NegativeInfinity};
{
    var ctrls = SlipConvergence_Droplet(a,b,degS,GridSeq,thetaS,lsS,sigmaS,hvapS);
    ctrls.ForEach(s => s.SessionName = "Stage1_" + s.SessionName);
    controls.AddRange(ctrls);
}

//stage 2
thetaS            = new double[] {80, 90};
lsS               = new double[] {0.0, 0.1, double.PositiveInfinity};
sigmaS                = new double[] {0.1};
hvapS                = new double[] {1e3};
{
    var ctrls = SlipConvergence_Droplet(a,b,degS,GridSeq,thetaS,lsS,sigmaS,hvapS);
    ctrls.ForEach(s => s.SessionName = "Stage2_" + s.SessionName);
    controls.AddRange(ctrls);
}

//stage 3
thetaS            = new double[] {80, 90};
lsS               = new double[] {0.0, 0.1, double.PositiveInfinity};
sigmaS                = new double[] {0.0};
hvapS                = new double[] {1e3};
{
    var ctrls = SlipConvergence_Droplet(a,b,degS,GridSeq,thetaS,lsS,sigmaS,hvapS);
    ctrls.ForEach(s => s.SessionName = "Stage3_" + s.SessionName);
    controls.AddRange(ctrls);
}

Console.WriteLine("# of controls: " + controls.Count());
int ctrlCount = controls.Count();

In [10]:
// // FOR TESTING IF THIS RUNS
// foreach(var C in controls){
// var C = controls.ElementAt(80);
// C.ImmediatePlotPeriod = 1;
// C.SuperSampling = 2;
// C.savetodb = false;
// C.PostprocessingModules.Clear();
// using(var solver = new XNSFE()){
//     solver.Init(C);
//     solver.RunSolverMode();
// }
// }


In [11]:
bool run      = true;
var bpc = BoSSSshell.GetDefaultQueue();

In [12]:
//BoSSSshell.WorkflowMgm.ResetProject(true, true, true, true);

In [13]:
if(BoSSSshell.WorkflowMgm.AllJobs.Count() > 0){
     BoSSSshell.WorkflowMgm.ResetProject();
}
var jobs = controls.Select(c => c.CreateJob()).ToArray();
jobs.ForEach(j => j.NumberOfThreads = 1);
jobs.ForEach(j => j.EnvironmentVars["BOSSS_ARG_7"] = j.NumberOfThreads.ToString());

In [ ]:
jobs.Activate();

In [46]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions.ForEach(s => display(s));

In [47]:
sessions.Count()

In [ ]:
// Für Vislt notwendig

//sessions.ForEach(session=> session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do());
// session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).WithShadowFields().Do();
// session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do();

In [ ]:
// Process.Start("explorer.exe", sessions.Third().GetSessionDirectory());

In [25]:
Plot2Ddata bndycond = new Plot2Ddata();

foreach(var theta in new double[] {90.0, 80.0}){
    double a = 0.816;
    double b = 0.784;
    double y_off = b / (Math.Sqrt(1 + (Math.Pow(a, 2) / Math.Pow(b, 2)) * Math.Pow((Math.Tan(theta/90.0*Math.PI/2.0)), 2)));
    // intersect with the x-axis
    double x0 = a * Math.Cos(Math.Asin(y_off / b));
    string[] lines;
    if(theta == 90.0) {
        lines = File.ReadAllLines("Temperature_boundarycond90.txt");
    } else if(theta == 80.0) {
        lines = File.ReadAllLines("Temperature_boundarycond80.txt");
    } else {
        throw new NotImplementedException();
    }

    double[] nodes = new double[lines.Length];
    double[] values = new double[lines.Length];
    for (int i = 0; i < lines.Length; i++) {
        string[] xy = lines[i].Split('\t');
        nodes[i] = Convert.ToDouble(xy[0]);
        values[i] = Convert.ToDouble(xy[1]);
    }
    double scale = x0 / nodes.Max();
    nodes = nodes.Select(n => scale * n).ToArray();

    bndycond = bndycond.Merge(new Plot2Ddata(nodes, values, theta.ToString()));
}


To plot the temperature b.c. in latex with tikz

In [ ]:
//bndycond.SaveToTextFile("bndycondT.txt");

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(14400);

In [ ]:
int count = BoSSSshell.wmg.Sessions.Count();
int success = BoSSSshell.wmg.Sessions.Where(s => s.SuccessfulTermination).Count();

if(count != ctrlCount || count != success){
    throw new ApplicationException("Not all simulations calculated or finished successful!");
}